# Subword Tokenization

In this exercise, we will learn how to train our own subword tokenizers with different algorithms: BPE and Unigram. We will use `sentencepiece`, a library from Google to help create our tokenizers.

## Ref:
https://github.com/google/sentencepiece/blob/master/python

## Setup

In [1]:
!wget https://github.com/Knight-H/thai-lm/raw/refs/heads/master/data/pra-apai-manee-ch1-50.txt
!wget https://github.com/Knight-H/thai-lm/raw/refs/heads/master/data/kratoo-40000000-40002000.jsonl

--2025-02-26 10:17:11--  https://github.com/Knight-H/thai-lm/raw/refs/heads/master/data/pra-apai-manee-ch1-50.txt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Knight-H/thai-lm/refs/heads/master/data/pra-apai-manee-ch1-50.txt [following]
--2025-02-26 10:17:13--  https://raw.githubusercontent.com/Knight-H/thai-lm/refs/heads/master/data/pra-apai-manee-ch1-50.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3231076 (3.1M) [application/octet-stream]
Saving to: â€˜pra-apai-manee-ch1-50.txtâ€™

pra-apai-manee-ch1- 100%[===================>]   3.08M  --.-KB/s    in 0.02s   

2025-02-26 10:17:1

## Code

In [2]:
import sentencepiece as spm
import io
import json

Load data

In [3]:
pantip_text = []
with open('kratoo-40000000-40002000.jsonl', 'r') as json_file:
    json_list = list(json_file)
    for json_str in json_list:
        result = json.loads(json_str)
        pantip_text.append(f"{result['title']}\n{result['content']}\n")
sum([len(t) for t in pantip_text])

1060318

In [4]:
with open("pra-apai-manee-ch1-50.txt") as f:
  pra_apai_manee_data = f.readlines()

In [5]:
sum([len(t) for t in pra_apai_manee_data])

1100605

In [6]:
pantip_train_text = pantip_text[:int(len(pantip_text)*0.8)]
pantip_test_text = pantip_text[int(len(pantip_text)*0.8):]

pam_train_text = pra_apai_manee_data[:int(len(pra_apai_manee_data)*0.8)] #pam = pra_apai_manee
pam_test_text = pra_apai_manee_data[int(len(pra_apai_manee_data)*0.8):]

## Run tokenizer training

The Python wrapper provides multiple APIs for training our tokenizers

1. `spm.SentencePieceTrainer.train(input='input.txt', model_prefix='m', vocab_size=vocab_size, model_type=model_type)`
  <br> This will output the tokenizer files `m.model` and `m.vocab` that can be later loaded into `SentencePieceProcessor`.
  <br><br>
2. `spm.SentencePieceTrainer.train(sentence_iterator=iterator, model_writer=obj_with_write_method, vocab_size=vocab_size, model_type=model_type)`
  <br> This method will require a file object e.g. `obj_with_write_method = io.BytesIO()`. The advantage of this method is you can run sentencepiece on environments that have limited access to the local file system. But you will still have to save the model file if you want to re-use the model else you will have to train it again.
<br><br>
3.  `spm.SentencePieceTrainer.train('--input=input.txt --model_prefix=m --vocab_size=vocab_size --model_type=model_type')`
<br> Same as no.1




### Unigram tokenizer

We are going to start with training a unigram tokenizer. You can use any method of training one. Make sure to set vocab_size to 1000.

In [7]:
## Train
with open("pam_train_text.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(pam_train_text))  # Ensure each sentence is on a new line

In [8]:
import sentencepiece as spm

# Define parameters
model_prefix = "unigram_pam_model"
vocab_size = 1000
model_type = "unigram"

# Train the unigram tokenizer using the saved input file
spm.SentencePieceTrainer.train(input="pam_train_text.txt", model_prefix=model_prefix, vocab_size=vocab_size, model_type=model_type)


### Q1 MCV

How many tokens did you get when tokenizing the following sentence with your unigram tokenizer: <br>
'à¸­à¸£à¸¸à¸“à¸ªà¸§à¸±à¸ªà¸”à¸´à¹Œ à¸‰à¸±à¸™à¹€à¸­à¸²à¸¡à¹€à¸«à¸ªà¸µà¸¡à¸²à¸«à¸²à¸¡ à¸ªà¸§à¸±à¸ªà¸”à¸µ à¸›à¸£à¸°à¹€à¸—à¸¨à¹„à¸—à¸¢à¸ªà¸šà¸²à¸¢à¸”à¸µà¹„à¸«à¸¡'

In [9]:
sp_pam = spm.SentencePieceProcessor()
sp_pam.load('unigram_pam_model.model')

True

In [10]:
len(sp_pam.encode('à¸­à¸£à¸¸à¸“à¸ªà¸§à¸±à¸ªà¸”à¸´à¹Œ à¸‰à¸±à¸™à¹€à¸­à¸²à¸¡à¹€à¸«à¸ªà¸µà¸¡à¸²à¸«à¸²à¸¡ à¸ªà¸§à¸±à¸ªà¸”à¸µ à¸›à¸£à¸°à¹€à¸—à¸¨à¹„à¸—à¸¢à¸ªà¸šà¸²à¸¢à¸”à¸µà¹„à¸«à¸¡', out_type=str))

29

### BPE Tokenizer

Now try training a BPE tokenizer.

In [11]:
import sentencepiece as spm

# Define parameters for BPE
model_prefix_bpe = "bpe_pam_model"
vocab_size = 1000
model_type_bpe = "bpe"

# Train the BPE tokenizer
spm.SentencePieceTrainer.train(input="pam_train_text.txt", model_prefix=model_prefix_bpe, vocab_size=vocab_size, model_type=model_type_bpe)

# This will generate bpe_model.model and bpe_model.vocab


### Q2 MCV

How many tokens did you get when tokenizing the following sentence with your BPE tokenizer: <br>
'à¸­à¸£à¸¸à¸“à¸ªà¸§à¸±à¸ªà¸”à¸´à¹Œ à¸‰à¸±à¸™à¹€à¸­à¸²à¸¡à¹€à¸«à¸ªà¸µà¸¡à¸²à¸«à¸²à¸¡ à¸ªà¸§à¸±à¸ªà¸”à¸µ à¸›à¸£à¸°à¹€à¸—à¸¨à¹„à¸—à¸¢à¸ªà¸šà¸²à¸¢à¸”à¸µà¹„à¸«à¸¡'

In [12]:
bpe = spm.SentencePieceProcessor()
bpe.load('bpe_pam_model.model')

True

In [13]:
len(bpe.encode('à¸­à¸£à¸¸à¸“à¸ªà¸§à¸±à¸ªà¸”à¸´à¹Œ à¸‰à¸±à¸™à¹€à¸­à¸²à¸¡à¹€à¸«à¸ªà¸µà¸¡à¸²à¸«à¸²à¸¡ à¸ªà¸§à¸±à¸ªà¸”à¸µ à¸›à¸£à¸°à¹€à¸—à¸¨à¹„à¸—à¸¢à¸ªà¸šà¸²à¸¢à¸”à¸µà¹„à¸«à¸¡', out_type=str))

28

These are some of your vocabs. Note that you will see "â–" (U+2581) in every type of tokenizer in SentencePiece since it makes it possible to perform detokenization \(unsplit your sentences\) without relying on language-specific resources.

In [14]:
unigram_vocabs = [sp_pam.id_to_piece(id) for id in range(sp_pam.get_piece_size())]
" | ".join(unigram_vocabs[:500])

'<unk> | <s> | </s> | â– | à¸² | à¹€ | à¸™ | à¸¡ | à¸¢ | à¸ | à¸£ | à¸§ | à¸” | à¸ª | à¸‡ | à¸š | à¸„ | à¸¡à¸² | à¸­ | à¸¥ | à¸ˆà¸° | à¸— | à¹ƒà¸«à¹‰ | à¸« | à¹„à¸› | à¹„à¸¡à¹ˆ | à¹ | à¸§à¹ˆà¸² | à¸ž | à¸¸ | à¸µ | à¹ | à¸¯ | à¸‚ | à¸Š | à¹€à¸›à¹‡à¸™ | à¸žà¸£à¸° | à¹‚ | à¸—à¸µà¹ˆ | à¹ƒà¸ˆ | â–à¸ˆà¸° | à¸ˆ | à¸° | à¸´ | à¸• | à¸à¹‡ | à¸­à¸¢à¸¹à¹ˆ | à¸› | à¹„à¸”à¹‰ | à¹ˆ | à¹„ | à¹€à¸‚à¹‰à¸² | à¸¹ | â–à¸žà¸£à¸° | à¹‰à¸² | à¸•à¸²à¸¡ | à¹ƒà¸™ | à¹‰ | â–à¹à¸¥à¹‰à¸§ | à¹€à¸«à¸¡à¸·à¸­à¸™ | à¸£à¸² | à¸¨ | à¹€à¸ˆà¹‰à¸² | à¹€à¸«à¹‡à¸™ | à¸¥à¸² | à¸à¸±à¸™ | à¸± | à¸«à¸² | à¸™à¸²à¸‡ | à¸—à¸£à¸‡ | à¸›à¸£à¸° | à¹Œ | à¸¢à¸² | à¸±à¸ | à¹à¸² | à¸‹ | à¸²à¸™ | à¸±à¸‡ | à¸‰ | à¸­à¸‡à¸„à¹Œ | à¸±à¸” | à¹à¸¥à¹‰à¸§ | à¸­à¸™ | à¸”à¸¹ | à¸– | à¸”à¹‰à¸§à¸¢ | à¸¡à¸µ | â–à¸ˆà¸¶à¸‡ | à¸™à¸µà¹‰ | à¹ˆà¸² | à¸œ | à¸™à¹‰à¸­à¸‡ | à¹à¸•à¹ˆ | à¸—à¹à¸² | â–à¸™à¸²à¸‡ | â–à¹ƒà¸«à¹‰ | à¸£à¸±à¸ | à¸žà¸µà¹ˆ | à¸„à¸´à¸” | à¸¥à¸¹à¸ | à¸žà¸² | à¸£à¸¹à¹‰ | à¸à¸²à¸£ | à¸à¸±à¸š | à¸±à¸™ | à¸«à¸™à¹‰à¸² | à¸à¸£à¸°

In [15]:
bpe_vocabs = [bpe.id_to_piece(id) for id in range(bpe.get_piece_size())]
" | ".join(bpe_vocabs[:500])

'<unk> | <s> | </s> | à¹‰à¸² | à¹ˆà¸² | à¸­à¸‡ | à¸£à¸° | à¹à¸² | à¸£à¸² | à¸­à¸¢ | à¹ˆà¸‡ | à¸¡à¸² | à¸ˆà¸° | à¸±à¸‡ | à¸±à¸™ | â–à¹€ | à¸²à¸¢ | à¹‰à¸§ | à¸±à¸š | à¸µà¹ˆ | à¸¡à¹ˆ | à¸­à¸™ | à¹ƒà¸« | à¸²à¸¡ | à¹‰à¸™ | à¹‡à¸™ | à¸žà¸£à¸° | à¸µà¸¢ | à¸²à¸‡ | à¸à¸¥ | à¹‰à¸‡ | à¸±à¸ | à¸«à¸™ | à¹ƒà¸«à¹‰ | à¹„à¸¡à¹ˆ | à¸«à¸¥ | à¹ˆà¸™ | à¸¶à¸‡ | â–à¹ | à¸—à¸± | à¸•à¸£ | à¸²à¸£ | à¹‰à¸­à¸‡ | à¹„à¸› | à¸´à¸” | à¸‚à¹‰à¸² | à¸§à¹ˆà¸² | à¸«à¸¡ | à¸„à¸£ | à¸·à¸­ | à¸¥à¹‰à¸§ | à¹€à¸› | à¹€à¸ª | à¸›à¸£à¸° | à¸²à¸™ | à¸±à¹ˆà¸‡ | â–à¹ | â–à¸¯ | à¸—à¸µà¹ˆ | à¸­à¸ | à¹€à¸¥ | à¸´à¸™ | à¹„à¸” | à¸žà¸¥ | à¸—à¸£ | à¸±à¸” | à¸™à¸²à¸‡ | à¸¶à¸ | à¹„à¸”à¹‰ | à¸¹à¹ˆ | â–à¸ˆà¸° | à¸„à¹Œ | à¸µà¹‰ | à¸žà¸£ | à¹€à¸›à¹‡à¸™ | à¸ªà¸¸ | à¸—à¸±à¹‰à¸‡ | à¸­à¸¡ | à¸±à¸¢ | à¹€à¸£ | à¸«à¹‡à¸™ | â–à¸ˆ | â–à¸žà¸£à¸° | à¸à¹‡ | à¹ƒà¸ˆ | à¸­à¸² | à¸·à¹ˆ | à¹ˆà¸²à¸‡ | à¸•à¹ˆ | à¸à¸£ | à¸´à¸‡ | à¸§à¸‡ | à¸§à¸™ | à¸·à¸­à¸™ | à¹€à¸ˆ | à¸¹à¹‰ | à¸µà¸¢à¸‡ | à¸­à¸¢à¸¹à¹ˆ | à¸£à¸£ | à¸•à¸²à¸¡ | â–à¸ž | à¹‰à¸§à¸¢ | à¸²à¸§

### User-defined symbols

Another important concept to know of is User-defined symbols. These special symbols are reserved for a special purpose \(e.g.\, the \<MASK\> token used in BERT) and will always be tokenized into one token.

Refer to the documentation for ways to add these special tokens to your tokenizer.

https://github.com/google/sentencepiece/blob/master/python

## Train another tokenizer on another domain

Now try training another unigram tokenizer on `pantip_text` and we will use it to compare with the unigram tokenizer we trained earlier.

In [16]:
## Train
with open("pantip_train_text.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(pantip_train_text))  # Ensure each sentence is on a new line

In [17]:
import sentencepiece as spm

# Define parameters
model_prefix = "unigram_pantip_model"
vocab_size = 1000
model_type = "unigram"

# Train the unigram tokenizer using the saved input file
spm.SentencePieceTrainer.train(input="pantip_train_text.txt", model_prefix=model_prefix, vocab_size=vocab_size, model_type=model_type)


In [18]:
pantip_unigram_tokenizer = spm.SentencePieceProcessor()
pantip_unigram_tokenizer.load('unigram_pantip_model.model')

True

In [19]:
import sentencepiece as spm

# Define parameters for BPE
model_prefix_bpe = "bpe_pantip_model"
vocab_size = 1000
model_type_bpe = "bpe"

# Train the BPE tokenizer
spm.SentencePieceTrainer.train(input="pantip_train_text.txt", model_prefix=model_prefix_bpe, vocab_size=vocab_size, model_type=model_type_bpe)

# This will generate bpe_model.model and bpe_model.vocab


In [20]:
pantip_bpe_tokenizer = spm.SentencePieceProcessor()
pantip_bpe_tokenizer.load('bpe_pantip_model.model')

True

## Analyse top tokens on different datasets

Use your tokenizers to tokenize the datasets and analyse your most common vocabularies (try 300-400 vocabs with len>1). Hint: tokenize your data and count the tokens.

In [21]:
from collections import defaultdict

In [22]:
from collections import Counter

token_pam_unigram = Counter(
    token for s in pam_test_text for token in sp_pam.encode(s, out_type=str) if len(token) > 1
)

top_tokens_unigram_pam = token_pam_unigram.most_common(300)
print(top_tokens_unigram_pam)

[('à¸¡à¸²', 666), ('à¹„à¸›', 657), ('à¹ƒà¸«à¹‰', 650), ('à¸ˆà¸°', 622), ('à¹„à¸¡à¹ˆ', 536), ('à¸§à¹ˆà¸²', 520), ('à¸žà¸£à¸°', 462), ('â–à¸ˆà¸°', 421), ('à¸—à¸µà¹ˆ', 417), ('à¹€à¸›à¹‡à¸™', 387), ('à¸£à¸²', 326), ('à¸¢à¸²', 321), ('à¸­à¸¢à¸¹à¹ˆ', 314), ('à¹ƒà¸ˆ', 313), ('à¹€à¸‚à¹‰à¸²', 297), ('à¸„à¸™', 295), ('à¹„à¸”à¹‰', 288), ('à¸¥à¸²', 282), ('à¸•à¸²à¸¡', 280), ('à¸à¹‡', 278), ('à¹ˆà¸²', 257), ('à¸²à¸™', 254), ('à¸à¸±à¸™', 253), ('â–à¹à¸¥à¹‰à¸§', 251), ('â–à¹ƒà¸«à¹‰', 251), ('à¸¡à¸µ', 245), ('à¸­à¸™', 245), ('à¹€à¸«à¸¡à¸·à¸­à¸™', 245), ('â–à¸žà¸£à¸°', 238), ('à¸›à¸£à¸°', 238), ('à¸•à¸£', 236), ('à¹ƒà¸™', 235), ('à¹€à¸ˆà¹‰à¸²', 235), ('à¹à¸²', 234), ('â–à¸™à¸²à¸‡', 231), ('à¹‰à¸²', 231), ('à¸žà¸¥', 230), ('à¸™à¹‰à¸­à¸‡', 227), ('à¸”à¸µ', 223), ('à¸—à¹à¸²', 217), ('à¸­à¸‡à¸„à¹Œ', 216), ('à¸±à¸š', 214), ('à¸žà¸µà¹ˆ', 213), ('à¸±à¸”', 209), ('â–à¸à¹ˆà¸²à¸¢', 208), ('à¸„à¸´à¸”', 206), ('à¸à¸±à¸š', 206), ('à¸žà¸²', 204), ('à¸”à¹‰à¸§à¸¢', 203), ('à¸™à¸²à¸¢', 203), ('à¹ˆà¸­', 203

In [23]:
token_pam_bpe = Counter(
    token for s in pam_test_text for token in bpe.encode(s, out_type=str) if len(token) > 1
)

top_tokens_bpe_pam = token_pam_bpe.most_common(300)
print(top_tokens_bpe_pam)

[('à¸¡à¸²', 678), ('à¹ƒà¸«à¹‰', 635), ('à¸ˆà¸°', 613), ('à¹„à¸¡à¹ˆ', 541), ('à¹„à¸›', 538), ('à¸žà¸£à¸°', 486), ('à¸§à¹ˆà¸²', 469), ('â–à¸¯', 455), ('â–à¹', 454), ('â–à¸ˆà¸°', 441), ('à¸²à¸™', 389), ('à¹€à¸›à¹‡à¸™', 387), ('à¸›à¸£à¸°', 381), ('à¸žà¸¥', 375), ('à¸—à¸µà¹ˆ', 353), ('à¸²à¸¢', 350), ('â–à¹€', 349), ('à¸à¸¥', 330), ('à¹ˆà¸²', 320), ('à¹ƒà¸ˆ', 313), ('à¸£à¸²', 307), ('à¸—à¸±à¹‰à¸‡', 301), ('à¸„à¸™', 298), ('à¸„à¸£', 293), ('à¸­à¸‡', 274), ('à¸­à¸¢à¸¹à¹ˆ', 265), ('à¸­à¸™', 262), ('à¹„à¸”à¹‰', 255), ('â–à¹à¸¥à¹‰à¸§', 251), ('à¸²à¸§', 250), ('à¸à¸±à¸™', 250), ('â–à¸žà¸£à¸°', 248), ('à¸•à¸²à¸¡', 248), ('à¸±à¸š', 243), ('â–à¸™à¸²à¸‡', 241), ('à¹€à¸ª', 241), ('à¹€à¸ˆà¹‰à¸²', 239), ('à¸±à¸”', 238), ('â–à¹ƒà¸«à¹‰', 236), ('à¹ƒà¸™', 235), ('à¸¥à¸²', 234), ('à¸•à¸£', 233), ('à¸à¸£', 230), ('à¸«à¸¥', 227), ('à¸”à¸µ', 226), ('â–à¸à¹ˆà¸²à¸¢', 224), ('à¸´à¸™', 223), ('à¹€à¸‚à¹‰à¸²', 222), ('à¸à¹‡', 222), ('à¹€à¸', 218), ('à¸¡à¸µ', 213), ('â–à¹„à¸¡à¹ˆ', 211), ('à¸±à¸™', 20

In [24]:
token_pantip_unigram = Counter(
    token for s in pantip_test_text for token in pantip_unigram_tokenizer.encode(s, out_type=str) if len(token) > 1
)

top_tokens_unigram_pantip = token_pantip_unigram.most_common(300)
print(top_tokens_unigram_pantip)

[('à¸—à¸µà¹ˆ', 1137), ('à¹„à¸”à¹‰', 666), ('à¹„à¸›', 659), ('à¸¡à¸²', 656), ('à¸ˆà¸°', 633), ('à¸à¹‡', 628), ('à¹€à¸£à¸²', 602), ('à¸§à¹ˆà¸²', 568), ('à¸¡à¸µ', 550), ('à¹„à¸¡à¹ˆ', 536), ('à¹€à¸›à¹‡à¸™', 522), ('à¸à¸²à¸£', 440), ('à¹ƒà¸«à¹‰', 405), ('à¸‚à¸­à¸‡', 397), ('à¹‰à¸²', 396), ('à¹€à¸¥à¸¢', 385), ('à¹ƒà¸™', 379), ('à¹à¸¥à¹‰à¸§', 366), ('à¸„à¹ˆà¸°', 343), ('à¸™à¸µà¹‰', 338), ('à¸„à¸£à¸±à¸š', 323), ('à¸à¸±à¸™', 296), ('à¸à¸±à¸š', 289), ('à¸”à¸µ', 284), ('à¸„à¸™', 273), ('à¸¡à¸±à¸™', 269), ('à¹€à¸‚à¸²', 260), ('â–à¹à¸•à¹ˆ', 260), ('à¸ˆà¸²à¸', 246), ('à¸¡à¸²à¸', 244), ('à¸­à¸¢à¸¹à¹ˆ', 244), ('à¹ˆà¸²', 219), ('â–à¹€à¸£à¸²', 213), ('à¸•à¹‰à¸­à¸‡', 209), ('à¸—à¹à¸²', 206), ('à¹ƒà¸Šà¹‰', 206), ('à¹à¸²', 198), ('à¸”à¹‰à¸§à¸¢', 197), ('à¸–à¸¶à¸‡', 190), ('à¹à¸•à¹ˆ', 188), ('â–à¹à¸¥à¸°', 180), ('à¸”à¸¹', 179), ('à¸„à¸§à¸²à¸¡', 179), ('à¹à¸¥à¸°', 179), ('à¸±à¸š', 176), ('à¸«à¸²', 173), ('à¹€à¸‚à¹‰à¸²', 172), ('à¸­à¸°à¹„à¸£', 169), ('à¹€à¸£à¸·à¹ˆà¸­à¸‡', 167), ('à¹„à¸¡à¹ˆà¹„à

In [25]:
token_pantip_bpe = Counter(
    token for s in pantip_test_text for token in pantip_bpe_tokenizer.encode(s, out_type=str) if len(token) > 1
)

top_tokens_bpe_pantip = token_pantip_bpe.most_common(300)
print(top_tokens_bpe_pantip)

[('à¸—à¸µà¹ˆ', 974), ('à¸¡à¸²', 662), ('à¹„à¸›', 634), ('à¹„à¸”à¹‰', 604), ('à¹„à¸¡à¹ˆ', 554), ('à¸§à¹ˆà¸²', 535), ('à¸¡à¸µ', 511), ('à¹€à¸£à¸²', 510), ('à¸ˆà¸°', 479), ('à¸à¹‡', 477), ('à¹€à¸›à¹‡à¸™', 447), ('â–à¹€', 431), ('à¹ƒà¸«à¹‰', 402), ('à¹à¸¥à¹‰à¸§', 366), ('à¸à¸²à¸£', 366), ('à¸‚à¸­à¸‡', 360), ('à¸„à¹ˆà¸°', 352), ('à¸™à¸µà¹‰', 341), ('à¹€à¸¥à¸¢', 331), ('à¸„à¸£à¸±à¸š', 327), ('à¹ƒà¸™', 325), ('à¹‰à¸²', 320), ('à¸”à¸µ', 306), ('à¸¡à¸²à¸', 284), ('à¸à¸±à¸™', 280), ('à¸„à¸™', 279), ('â–à¹à¸•à¹ˆ', 260), ('à¸­à¸‡', 258), ('à¸—à¹à¸²', 255), ('à¸à¸±à¸š', 250), ('â–à¹€à¸£à¸²', 243), ('à¸­à¸¢à¸¹à¹ˆ', 243), ('à¹à¸•à¹ˆ', 224), ('à¸­à¸™', 224), ('à¸„à¸§à¸²à¸¡', 221), ('â–à¸ª', 215), ('à¸­à¸', 214), ('à¸›à¸£à¸°', 211), ('à¸¡à¸±à¸™', 210), ('à¸ˆà¸²à¸', 209), ('à¸±à¸™', 205), ('à¸´à¸™', 200), ('à¹à¸²', 200), ('à¹€à¸', 198), ('à¸²à¸¢', 197), ('à¸”à¹‰à¸§à¸¢', 197), ('à¸±à¸‡', 191), ('à¹ƒà¸ˆ', 189), ('à¸–à¸¶à¸‡', 189), ('à¹ˆà¸²', 187), ('à¸±à¸š', 186), ('à¹‰à¸™', 185), ('à¸²à¸

### To answer
What are some notable differences you see between the two vocabs?

Write your answer below.

> à¸„à¸§à¸²à¸¡à¸–à¸µà¹ˆà¸‚à¸­à¸‡ token à¹ƒà¸™ BPE à¹à¸¥à¸° unigram à¹„à¸¡à¹ˆà¹€à¸—à¹ˆà¸²à¸à¸±à¸™ à¸—à¸±à¹‰à¸‡à¹ƒà¸™ pantip à¹à¸¥à¸° à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ

> à¸ à¸²à¸žà¸£à¸§à¸¡à¸—à¸±à¹‰à¸‡ unigram à¹à¸¥à¸° bpe à¹ƒà¸«à¹‰à¸œà¸¥à¸„à¸¥à¹‰à¸²à¸¢à¸à¸±à¸™




## Using tokenizer across domains

One problem you may face is your dataset is very specialized. In that case the tokenizer trained on a general domain may not perform as good as it should when used on your dataset.

Next you will try using tokenizers trained on one general domain (on Pantip) and use it on a specialized domain (à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ) and vice versa.

### Q3 MCV

What percentage increase do you observe when tokenizing the whole à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ dataset with a tokenizer trained on Pantip compared to the one trained on à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ.

In [26]:
import sentencepiece as spm

# Train tokenizer on à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ dataset
from_pra_apai = spm.SentencePieceProcessor()
spm.SentencePieceTrainer.train(
    sentence_iterator=iter(pam_train_text),
    model_prefix='from_pra_apai',
    vocab_size=1000
)
from_pra_apai.load('from_pra_apai.model')

# Train tokenizer on Pantip dataset
from_pantip = spm.SentencePieceProcessor()
spm.SentencePieceTrainer.train(
    sentence_iterator=iter(pantip_train_text),
    model_prefix='from_pantip',
    vocab_size=1000
)
from_pantip.load('from_pantip.model')

True

In [27]:
# Convert à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ dataset to a single string
pra_apai_text = ' '.join(pra_apai_manee_data)

# Tokenization count using à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ tokenizer
pra_apai_token_count = len(from_pra_apai.encode(pra_apai_text, out_type=str))
print(f'Tokens from à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ tokenizer: {pra_apai_token_count}')

# Tokenization count using Pantip tokenizer
pantip_token_count = len(from_pantip.encode(pra_apai_text, out_type=str))
print(f'Tokens from Pantip tokenizer: {pantip_token_count}')

# Calculate percentage increase
percentage_increase = ((pantip_token_count - pra_apai_token_count) / pra_apai_token_count) * 100
print(f'Percentage increase: {percentage_increase:.2f}%')

Tokens from à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ tokenizer: 446143
Tokens from Pantip tokenizer: 639890
Percentage increase: 43.43%


### Q4 MCV

What percentage increase do you observe when tokenizing the whole Pantip dataset with a tokenizer trained on à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ compared to the one trained on Pantip.

In [28]:
# Convert Pantip dataset to a single string
pantip_text = ' '.join(pantip_train_text)

# Tokenization count using Pantip tokenizer
pantip_token_count_original = len(from_pantip.encode(pantip_text, out_type=str))
print(f'Tokens from Pantip tokenizer: {pantip_token_count_original}')

# Tokenization count using à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ tokenizer
pra_apai_token_count_on_pantip = len(from_pra_apai.encode(pantip_text, out_type=str))
print(f'Tokens from à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ tokenizer: {pra_apai_token_count_on_pantip}')

# Calculate percentage increase
percentage_increase_pantip = ((pra_apai_token_count_on_pantip - pantip_token_count_original) / pantip_token_count_original) * 100
print(f'Percentage increase: {percentage_increase_pantip:.2f}%')

Tokens from Pantip tokenizer: 365240
Tokens from à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ tokenizer: 413760
Percentage increase: 13.28%


### To answer
Why do you think the number of tokens tokenized by the general tokenizer (the one trained on Pantip) has a higher percentage increase compared to the number of tokens tokenized by the specialized tokenizer? (Hint: we fixed vocab size.)



1.   à¹„à¸¡à¹ˆà¸£à¸¹à¹‰à¸ˆà¸±à¸à¸„à¸³à¹€à¸‰à¸žà¸²à¸°à¸—à¸²à¸‡ â€“ à¸„à¸³à¸—à¸µà¹ˆà¸žà¸šà¸šà¹ˆà¸­à¸¢à¹ƒà¸™ à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ à¸­à¸²à¸ˆà¹„à¸¡à¹ˆà¸¡à¸µà¸­à¸¢à¸¹à¹ˆà¹ƒà¸™à¹‚à¸¡à¹€à¸”à¸¥à¸‚à¸­à¸‡ Pantip à¸—à¸³à¹ƒà¸«à¹‰à¸•à¹‰à¸­à¸‡à¹à¸šà¹ˆà¸‡à¹€à¸›à¹‡à¸™à¸«à¸¥à¸²à¸¢ token
2.   à¸‚à¸™à¸²à¸”à¸„à¸³à¸¨à¸±à¸žà¸—à¹Œà¸–à¸¹à¸à¸ˆà¸³à¸à¸±à¸” â€“ à¹à¸¡à¹‰à¸ˆà¸°à¸¡à¸µà¸‚à¸™à¸²à¸”à¸žà¸ˆà¸™à¸²à¸™à¸¸à¸à¸£à¸¡à¹€à¸—à¹ˆà¸²à¸à¸±à¸™ (1,000 à¸„à¸³) à¹à¸•à¹ˆ à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ Tokenizer à¸–à¸¹à¸à¸­à¸­à¸à¹à¸šà¸šà¸¡à¸²à¹ƒà¸«à¹‰à¹€à¸«à¸¡à¸²à¸°à¸ªà¸¡à¸à¸±à¸šà¸ à¸²à¸©à¸²à¸‚à¸­à¸‡à¸¡à¸±à¸™à¸¡à¸²à¸à¸à¸§à¹ˆà¸²
3. à¸„à¸§à¸²à¸¡à¹à¸•à¸à¸•à¹ˆà¸²à¸‡à¸‚à¸­à¸‡à¸ à¸²à¸©à¸² â€“ à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ à¹€à¸›à¹‡à¸™à¸ à¸²à¸©à¸²à¹€à¸à¹ˆà¸² à¸¡à¸µà¸£à¸¹à¸›à¹à¸šà¸šà¸§à¸£à¸£à¸“à¸à¸£à¸£à¸¡à¸—à¸µà¹ˆà¹à¸•à¸à¸•à¹ˆà¸²à¸‡à¸ˆà¸²à¸à¸ à¸²à¸©à¸²à¸žà¸¹à¸”à¹ƒà¸™ Pantip à¸—à¸³à¹ƒà¸«à¹‰ Tokenizer à¸—à¸±à¹ˆà¸§à¹„à¸›à¹à¸šà¹ˆà¸‡à¸„à¸³à¹„à¸”à¹‰à¹„à¸¡à¹ˆà¸”à¸µ



## The effect on language models

Next, we will see the effect of using "cross-domain" tokenizers on Language models.

### Setup
We are going to reuse the code from the last assignment

In [29]:
!pip install lightning

     â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 40.4/40.4 kB 3.3 MB/s eta 0:00:00
   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 815.2/815.2 kB 51.3 MB/s eta 0:00:00
   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 13.8/13.8 MB 102.9 MB/s eta 0:00:00
   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 24.6/24.6 MB 83.8 MB/s eta 0:00:00
   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 883.7/883.7 kB 52.8 MB/s eta 0:00:00
   â”â”â”â”â”â”â”â”â”â”â”â”

In [30]:
import itertools
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import lightning as L
from tqdm import tqdm
import numpy as np

In [31]:
class TextDataset(Dataset):
  def __init__(self, data, tokenizer, seq_len = 128):

    token_ids = [tokenizer.encode(d, add_bos=True, add_eos=True) for d in data]
    flatten_token_ids = list(itertools.chain(*token_ids))
    encoded = torch.LongTensor(flatten_token_ids)

    left_over = len(encoded) % seq_len
    encoded = encoded[:len(encoded)-left_over]
    self.encoded = encoded.view(-1, seq_len)

  def __getitem__(self, idx):
    return self.encoded[idx]

  def __len__(self):
    return len(self.encoded)

In [33]:
class LSTM(L.LightningModule):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate, learning_rate, criterion):

        super().__init__()

        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim
        self.vocab_size=vocab_size

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                    dropout=dropout_rate, batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.learning_rate = learning_rate
        self.criterion = criterion

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        output, (hidden, cell) = self.lstm(embedded)
        output = self.dropout(output)
        output = self.fc(output)
        return output

    def training_step(self, batch, batch_idx):

        src = batch[:, :-1]
        target = batch[:, 1:]
        prediction = self(src)
        prediction = prediction.reshape(-1, self.vocab_size)
        target = target.reshape(-1)
        loss = self.criterion(prediction, target)
        self.log("train_loss", loss)
        return loss

    def test_step(self, batch, batch_idx, dataloader_idx=0):

        src = batch[:, :-1]
        target = batch[:, 1:]
        with torch.no_grad():
          prediction = self(src)
        prediction = prediction.reshape(-1, self.vocab_size)
        target = target.reshape(-1)
        loss = self.criterion(prediction, target)
        self.log("test_loss", loss)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.learning_rate)

In [34]:
vocab_size = sp_pam.get_piece_size()
embedding_dim = 200
hidden_dim = 512
num_layers = 3
dropout_rate = 0.2
lr = 1e-3
criterion = nn.CrossEntropyLoss()
train_batch_size = 64
test_batch_size = 128

### Training

<a name="no1"></a>
#### 1. Training on Pantip data with Pantip tokenizer

In [35]:
sp_pantip = spm.SentencePieceProcessor(model_file='unigram_pantip_model.model')
sp_pam = spm.SentencePieceProcessor(model_file='unigram_pam_model.model')

In [36]:
trainer = L.Trainer(
    max_epochs=10,
    deterministic=True
)
model = LSTM(vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate, lr, criterion)

pantip_train_dataset = TextDataset(pantip_train_text, sp_pantip)
pantip_train_loader = DataLoader(pantip_train_dataset, batch_size = train_batch_size, shuffle = True)

pantip_test_dataset = TextDataset(pantip_test_text, sp_pantip)
pantip_test_loader = DataLoader(pantip_test_dataset, batch_size = test_batch_size, shuffle = False)

pam_train_dataset = TextDataset(pam_train_text, sp_pantip)
pam_train_loader = DataLoader(pam_train_dataset, batch_size = train_batch_size, shuffle = True)

pam_test_dataset = TextDataset(pam_test_text, sp_pantip)
pam_test_loader = DataLoader(pam_test_dataset, batch_size = test_batch_size, shuffle = False)

trainer.fit(model, train_dataloaders=pantip_train_loader)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | embedding | Embedding        | 200 K  | train
1 | lstm      | LSTM             | 5.7 M  | train
2 | dropout   | Dropout          | 0      | train
3 | fc        | Linear           | 513 K  | train
4 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
6.4 M     Trainable params
0         Non-trainable params
6.4 M     Tot

Training: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


In [37]:
test_result = trainer.test(model, dataloaders=[pantip_train_loader, pam_train_loader, pantip_test_loader,pam_test_loader], verbose=False)

print(f"Perplexity on Pantip train set is:\t{np.exp(test_result[0]['test_loss/dataloader_idx_0'])}")
print(f"Perplexity on Pra apai manee train set is:\t{np.exp(test_result[1]['test_loss/dataloader_idx_1'])}")
print(f"Perplexity on Pantip test set is:\t{np.exp(test_result[2]['test_loss/dataloader_idx_2'])}")
print(f"Perplexity on Pra apai manee test set is:\t{np.exp(test_result[3]['test_loss/dataloader_idx_3'])}")

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:476: Your `test_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.


Testing: |          | 0/? [00:00<?, ?it/s]

Perplexity on Pantip train set is:	85.47244360249539
Perplexity on Pra apai manee train set is:	114.38507620423142
Perplexity on Pantip test set is:	111.51977474910314
Perplexity on Pra apai manee test set is:	116.4923230890615


<a name="no2"></a>
#### 2. Training on Pantip data with Pra apai manee tokenizer

In [38]:
trainer = L.Trainer(
    max_epochs=10,
    deterministic=True
)
model = LSTM(vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate, lr, criterion)

pantip_train_dataset = TextDataset(pantip_train_text, sp_pam)
pantip_train_loader = DataLoader(pantip_train_dataset, batch_size = train_batch_size, shuffle = True)

pantip_test_dataset = TextDataset(pantip_test_text, sp_pam)
pantip_test_loader = DataLoader(pantip_test_dataset, batch_size = test_batch_size, shuffle = False)

pam_train_dataset = TextDataset(pam_train_text, sp_pam)
pam_train_loader = DataLoader(pam_train_dataset, batch_size = train_batch_size, shuffle = True)

pam_test_dataset = TextDataset(pam_test_text, sp_pam)
pam_test_loader = DataLoader(pam_test_dataset, batch_size = test_batch_size, shuffle = False)

trainer.fit(model, train_dataloaders=pantip_train_loader)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | embedding | Embedding        | 200 K  | train
1 | lstm      | LSTM             | 5.7 M  | train
2 | dropout   | Dropout          | 0      | train
3 | fc        | Linear           | 513 K  | train
4 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
6.4 M     Trainable params
0         Non-trainable params
6.4 M     Tot

Training: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


In [39]:
test_result = trainer.test(model, dataloaders=[pantip_train_loader, pam_train_loader, pantip_test_loader,pam_test_loader], verbose=False)

print(f"Perplexity on Pantip train set is:\t{np.exp(test_result[0]['test_loss/dataloader_idx_0'])}")
print(f"Perplexity on Pra apai manee train set is:\t{np.exp(test_result[1]['test_loss/dataloader_idx_1'])}")
print(f"Perplexity on Pantip test set is:\t{np.exp(test_result[2]['test_loss/dataloader_idx_2'])}")
print(f"Perplexity on Pra apai manee test set is:\t{np.exp(test_result[3]['test_loss/dataloader_idx_3'])}")

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

Perplexity on Pantip train set is:	39.343671966196894
Perplexity on Pra apai manee train set is:	442.3274975372621
Perplexity on Pantip test set is:	49.26134949527618
Perplexity on Pra apai manee test set is:	415.05098778524894


#### To answer

The perplexity numbers should indicate that:
1. Training the LM with Pra apai manee tokenizer on Pantip (no. [2](#no2)) results in overfitting to Pantip and poor generalization to the Pra apai manee dataset.
2. However using the Pantip tokenizer (no. [1](#no1)) results in a much better generalization.

Try and come up with some reasons for the results above. <br>
Hint:
1. think about "general" vocabs and domain-specific vocabs.
2. what do you think happens to the model when the token ids become longer.



* à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ Tokenizer (no. 2) à¸—à¸³à¹ƒà¸«à¹‰à¹‚à¸¡à¹€à¸”à¸¥ overfit à¸à¸±à¸š Pantip à¹à¸¥à¸° generalize à¹„à¸”à¹‰à¹à¸¢à¹ˆà¸¡à¸²à¸à¸šà¸™ à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ à¹€à¸žà¸£à¸²à¸°à¸¡à¸±à¸™à¹„à¸¡à¹ˆà¹„à¸”à¹‰à¸­à¸­à¸à¹à¸šà¸šà¸¡à¸²à¹ƒà¸«à¹‰à¸£à¸­à¸‡à¸£à¸±à¸šà¸ à¸²à¸©à¸²à¸žà¸¹à¸”à¸—à¸±à¹ˆà¸§à¹„à¸›
* à¸à¸²à¸£à¹ƒà¸Šà¹‰ Tokenizer à¸—à¸µà¹ˆà¹„à¸¡à¹ˆà¹€à¸«à¸¡à¸²à¸°à¸ªà¸¡à¸—à¸³à¹ƒà¸«à¹‰ token IDs à¸¢à¸²à¸§à¸‚à¸¶à¹‰à¸™ à¹‚à¸¡à¹€à¸”à¸¥à¸•à¹‰à¸­à¸‡à¸›à¸£à¸°à¸¡à¸§à¸¥à¸œà¸¥à¸„à¸³à¸—à¸µà¹ˆà¸–à¸¹à¸à¹à¸šà¹ˆà¸‡à¹€à¸›à¹‡à¸™à¸Šà¸´à¹‰à¸™à¹€à¸¥à¹‡à¸ à¹† à¹‚à¸”à¸¢à¹„à¸¡à¹ˆà¸ˆà¸³à¹€à¸›à¹‡à¸™ à¸ªà¹ˆà¸‡à¸œà¸¥à¹ƒà¸«à¹‰à¸à¸²à¸£à¹€à¸£à¸µà¸¢à¸™à¸£à¸¹à¹‰à¹„à¸¡à¹ˆà¸¡à¸µà¸›à¸£à¸°à¸ªà¸´à¸—à¸˜à¸´à¸ à¸²à¸ž
* Pantip Tokenizer (no. 1) generalize à¹„à¸”à¹‰à¸”à¸µà¸à¸§à¹ˆà¸² à¹€à¸žà¸£à¸²à¸°à¸¡à¸µà¸„à¸³à¸¨à¸±à¸žà¸—à¹Œà¸—à¸µà¹ˆà¸„à¸£à¸­à¸šà¸„à¸¥à¸¸à¸¡à¹à¸¥à¸°à¹€à¸«à¸¡à¸²à¸°à¸à¸±à¸šà¸ à¸²à¸©à¸²à¸ªà¸¡à¸±à¸¢à¹ƒà¸«à¸¡à¹ˆà¸¡à¸²à¸à¸à¸§à¹ˆà¸² à¸—à¸³à¹ƒà¸«à¹‰ Perplexity à¸‚à¸­à¸‡à¸Šà¸¸à¸”à¸‚à¹‰à¸­à¸¡à¸¹à¸¥à¸•à¹ˆà¸²à¸‡à¸à¸±à¸™à¹„à¸¡à¹ˆà¸¡à¸²à¸




<a name="no3"></a>
#### 3. Training on Pra apai manee data with Pantip tokenizer


In [40]:
trainer = L.Trainer(
    max_epochs=10,
    deterministic=True
)
model = LSTM(vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate, lr, criterion)

pantip_train_dataset = TextDataset(pantip_train_text, sp_pantip)
pantip_train_loader = DataLoader(pantip_train_dataset, batch_size = train_batch_size, shuffle = True)

pantip_test_dataset = TextDataset(pantip_test_text, sp_pantip)
pantip_test_loader = DataLoader(pantip_test_dataset, batch_size = test_batch_size, shuffle = False)

pam_train_dataset = TextDataset(pam_train_text, sp_pantip)
pam_train_loader = DataLoader(pam_train_dataset, batch_size = train_batch_size, shuffle = True)

pam_test_dataset = TextDataset(pam_test_text, sp_pantip)
pam_test_loader = DataLoader(pam_test_dataset, batch_size = test_batch_size, shuffle = False)

trainer.fit(model, train_dataloaders=pam_train_loader)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | embedding | Embedding        | 200 K  | train
1 | lstm      | LSTM             | 5.7 M  | train
2 | dropout   | Dropout          | 0      | train
3 | fc        | Linear           | 513 K  | train
4 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
6.4 M     Trainable params
0         Non-trainable params
6.4 M     Tot

Training: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


In [41]:
test_result = trainer.test(model, dataloaders=[pantip_train_loader, pam_train_loader, pantip_test_loader,pam_test_loader], verbose=False)

print(f"Perplexity on Pantip train set is:\t{np.exp(test_result[0]['test_loss/dataloader_idx_0'])}")
print(f"Perplexity on Pra apai manee train set is:\t{np.exp(test_result[1]['test_loss/dataloader_idx_1'])}")
print(f"Perplexity on Pantip test set is:\t{np.exp(test_result[2]['test_loss/dataloader_idx_2'])}")
print(f"Perplexity on Pra apai manee test set is:\t{np.exp(test_result[3]['test_loss/dataloader_idx_3'])}")

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

Perplexity on Pantip train set is:	3408.2298370046524
Perplexity on Pra apai manee train set is:	41.59302367798346
Perplexity on Pantip test set is:	2791.309418637445
Perplexity on Pra apai manee test set is:	44.6381658475185


<a name="no4"></a>
#### 4. Training on Pra apai manee data with Pra apai manee tokenizer




In [42]:
trainer = L.Trainer(
    max_epochs=10,
    deterministic=True
)
model = LSTM(vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate, lr, criterion)

pantip_train_dataset = TextDataset(pantip_train_text, sp_pam)
pantip_train_loader = DataLoader(pantip_train_dataset, batch_size = train_batch_size, shuffle = True)

pantip_test_dataset = TextDataset(pantip_test_text, sp_pam)
pantip_test_loader = DataLoader(pantip_test_dataset, batch_size = test_batch_size, shuffle = False)

pam_train_dataset = TextDataset(pam_train_text, sp_pam)
pam_train_loader = DataLoader(pam_train_dataset, batch_size = train_batch_size, shuffle = True)

pam_test_dataset = TextDataset(pam_test_text, sp_pam)
pam_test_loader = DataLoader(pam_test_dataset, batch_size = test_batch_size, shuffle = False)

trainer.fit(model, train_dataloaders=pam_train_loader)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | embedding | Embedding        | 200 K  | train
1 | lstm      | LSTM             | 5.7 M  | train
2 | dropout   | Dropout          | 0      | train
3 | fc        | Linear           | 513 K  | train
4 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
6.4 M     Trainable params
0         Non-trainable params
6.4 M     Tot

Training: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


In [43]:
test_result = trainer.test(model, dataloaders=[pantip_train_loader, pam_train_loader, pantip_test_loader,pam_test_loader], verbose=False)

print(f"Perplexity on Pantip train set is:\t{np.exp(test_result[0]['test_loss/dataloader_idx_0'])}")
print(f"Perplexity on Pra apai manee train set is:\t{np.exp(test_result[1]['test_loss/dataloader_idx_1'])}")
print(f"Perplexity on Pantip test set is:\t{np.exp(test_result[2]['test_loss/dataloader_idx_2'])}")
print(f"Perplexity on Pra apai manee test set is:\t{np.exp(test_result[3]['test_loss/dataloader_idx_3'])}")

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

Perplexity on Pantip train set is:	541.5097731131742
Perplexity on Pra apai manee train set is:	76.9741108748581
Perplexity on Pantip test set is:	530.6975818113015
Perplexity on Pra apai manee test set is:	85.28988697364925


#### To answer

The perplexity numbers should indicate that:
1. Both LM overfits on Pra apai manee data and performs really bad on Pantip data.
2. However using the Pra apai manee tokenizer (no. [4](#no4)) results in a  better generalization than the Pantip tokenizer(no. [3](#no3)).

Try and come up with some reasons for the results above. <br>

* à¸—à¸±à¹‰à¸‡à¸ªà¸­à¸‡à¹‚à¸¡à¹€à¸”à¸¥ overfit à¸à¸±à¸š à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ à¹€à¸™à¸·à¹ˆà¸­à¸‡à¸ˆà¸²à¸à¸–à¸¹à¸à¸à¸¶à¸à¸šà¸™à¸Šà¸¸à¸”à¸‚à¹‰à¸­à¸¡à¸¹à¸¥à¹€à¸”à¸µà¸¢à¸§à¸à¸±à¸™ à¸—à¸³à¹ƒà¸«à¹‰ Perplexity à¸•à¹ˆà¸³à¸šà¸™ à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ à¹à¸•à¹ˆà¸ªà¸¹à¸‡à¸¡à¸²à¸à¸šà¸™ Pantip
* Pantip Tokenizer (no. 3) à¸•à¸±à¸”à¸„à¸³à¹„à¸”à¹‰à¹„à¸¡à¹ˆà¸”à¸µ à¸ªà¸³à¸«à¸£à¸±à¸š à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ à¸—à¸³à¹ƒà¸«à¹‰à¹‚à¸¡à¹€à¸”à¸¥à¹€à¸‚à¹‰à¸²à¹ƒà¸ˆà¸‚à¹‰à¸­à¸¡à¸¹à¸¥à¸œà¸´à¸” à¸ªà¹ˆà¸‡à¸œà¸¥à¹ƒà¸«à¹‰ Perplexity à¸ªà¸¹à¸‡à¸‚à¸¶à¹‰à¸™à¸¡à¸²à¸à¹€à¸¡à¸·à¹ˆà¸­à¹ƒà¸Šà¹‰à¸à¸±à¸š Pantip
* à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ Tokenizer (no. 4) à¹€à¸«à¸¡à¸²à¸°à¸ªà¸¡à¸à¸§à¹ˆà¸² à¹€à¸žà¸£à¸²à¸°à¸•à¸±à¸”à¸„à¸³à¹„à¸”à¹‰à¸•à¸£à¸‡à¸à¸±à¸šà¹‚à¸„à¸£à¸‡à¸ªà¸£à¹‰à¸²à¸‡à¸‚à¸­à¸‡ à¸žà¸£à¸°à¸­à¸ à¸±à¸¢à¸¡à¸“à¸µ à¸Šà¹ˆà¸§à¸¢à¹ƒà¸«à¹‰à¹‚à¸¡à¹€à¸”à¸¥ generalize à¹„à¸”à¹‰à¸”à¸µà¸à¸§à¹ˆà¸² à¹à¸¡à¹‰à¸¢à¸±à¸‡à¸—à¸³à¸‡à¸²à¸™à¹à¸¢à¹ˆà¸šà¸™ Pantip